In [1]:
import json, os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES     = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR, N_MELS, TIME_FRAMES = 16000, 64, 128

with open("audioset_index.json") as f:
    IDX = json.load(f)

TRIGGER_MAP = {
    "/m/03qc9zr":"Scream","/m/07p6fty":"Shout","/t/dd00135":"Shout",
    "/m/0463cq4":"Crying","/t/dd00002":"Crying","/m/014zdl":"Explosion",
    "/m/0g6b5":"Explosion","/m/032s66":"Gunshot","/m/039jq":"Glass",
    "/m/03kmc9":"Siren","/m/04qvtq":"Siren","/m/012n7d":"Siren",
    "/m/012ndj":"Siren","/m/07pp_mv":"Alarm","/m/02mfyn":"Alarm",
    "/m/01y3hg":"Alarm","/m/0c3f7m":"Alarm",
}
tids = set(TRIGGER_MAP.keys())

def load_seg(path):
    d = pd.read_csv(path, skiprows=3, header=None,
                    names=["ytid","start_s","end_s","labels"],
                    quotechar='"', skipinitialspace=True, comment="#")
    d["ytid"] = d["ytid"].str.strip()
    return d[d["ytid"].isin(IDX)]

def tag(d):
    d = d.copy()
    for c in CLASSES:
        cids = [a for a,v in TRIGGER_MAP.items() if v==c]
        d[c] = d["labels"].apply(lambda x: 1 if any(a in x for a in cids) else 0)
    return d

print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(f"Index: {len(IDX):,} clips")

unb = tag(load_seg("unbalanced_train_segments.csv"))
bal = tag(load_seg("balanced_train_segments.csv"))
evl = tag(load_seg("eval_segments.csv"))
print(f"unbalanced={len(unb):,}  balanced={len(bal):,}  eval={len(evl):,}")

Device: cuda | GPU: NVIDIA RTX A4000
Index: 1,951,571 clips
unbalanced=1,912,134  balanced=20,550  eval=18,887


In [2]:
TARGET = 1200   # per class cap

# ── TRAIN/VAL POOL: unbalanced + balanced ────────────────────────
pool = pd.concat([unb, bal], ignore_index=True).drop_duplicates(subset="ytid")
print(f"Train/val pool: {len(pool):,} clips\n")

sampled = []
for c in CLASSES:
    cd = pool[pool[c] == 1]
    n  = min(TARGET, len(cd))
    sampled.append(cd.sample(n=n, random_state=42))
    print(f"  {c:12s}: {n:5d}   (available {len(cd):6,})")

pos = pd.concat(sampled).drop_duplicates(subset="ytid")
print(f"\nUnique positives: {len(pos):,}")

# ── HARD NEGATIVES ───────────────────────────────────────────────
HN = {"/m/09x0r","/m/04rlf","/m/07qmpdm","/m/03qtwd","/m/0261r1"}
def clean_neg(s): return len(set(s.strip('"').split(",")) & tids) == 0
def hard_neg(s):  return bool(set(s.strip('"').split(",")) & HN)

neg   = pool[~pool["ytid"].isin(set(pos["ytid"])) & pool["labels"].apply(clean_neg)]
hn_p  = neg[neg["labels"].apply(hard_neg)]
n_hn  = min(int(len(pos)*0.30), len(hn_p))
hard  = hn_p.sample(n=n_hn, random_state=42).copy()
hard[CLASSES] = 0
print(f"Hard negatives: {len(hard):,}")

# ── TRAIN / VAL 85/15 ────────────────────────────────────────────
allp   = pd.concat([pos, hard], ignore_index=True).sample(frac=1, random_state=42)
tr, va = train_test_split(allp, test_size=0.15, random_state=42)

# ── TEST = eval_segments (held out) ──────────────────────────────
ev_pos = evl[evl[CLASSES].sum(axis=1) > 0]
ev_neg = evl[(evl[CLASSES].sum(axis=1) == 0) & evl["labels"].apply(hard_neg)]
ev_neg = ev_neg.sample(n=min(int(len(ev_pos)*0.30), len(ev_neg)), random_state=42).copy()
te     = pd.concat([ev_pos, ev_neg], ignore_index=True).sample(frac=1, random_state=42)

for nm, d in [("train",tr), ("val",va), ("test",te)]:
    d.to_csv(f"audioset_v2_{nm}.csv", index=False)

print(f"\n{'Split':8s} {'clips':>7}")
print("-"*60)
for nm, d in [("train",tr), ("val",va), ("test",te)]:
    cs = " ".join(f"{c[:3]}:{d[c].sum():4d}" for c in CLASSES)
    print(f"{nm:8s} {len(d):7,}  {cs}")

print(f"\nTotal: {len(tr)+len(va)+len(te):,} clips   (v1 was 2,999 → {(len(tr)+len(va)+len(te))/2999:.1f}x)")
print("Test = eval_segments, standard AudioSet protocol (never in training).")

Train/val pool: 1,932,684 clips

  Scream      :  1063   (available  1,063)
  Shout       :  1200   (available  1,848)
  Crying      :  1200   (available  3,148)
  Explosion   :  1200   (available  4,870)
  Gunshot     :  1200   (available  3,732)
  Glass       :   740   (available    740)
  Siren       :  1200   (available 11,452)
  Alarm       :  1200   (available  2,166)

Unique positives: 8,980
Hard negatives: 2,694

Split      clips
------------------------------------------------------------
train      9,922  Scr: 912 Sho:1009 Cry:1009 Exp:1019 Gun:1006 Gla: 645 Sir:1091 Ala:1040
val        1,752  Scr: 151 Sho: 194 Cry: 199 Exp: 187 Gun: 195 Gla:  95 Sir: 185 Ala: 164
test       1,249  Scr:  52 Sho: 113 Cry:  95 Exp: 108 Gun: 160 Gla:  57 Sir: 204 Ala: 196

Total: 12,923 clips   (v1 was 2,999 → 4.3x)
Test = eval_segments, standard AudioSet protocol (never in training).


In [3]:
class AudioSetDS(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.df["ytid"] = self.df["ytid"].str.strip()
        self.df = self.df[self.df["ytid"].isin(IDX)].reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row    = self.df.iloc[i]
        labels = np.array([row[c] for c in CLASSES], dtype=np.float32)
        y, _   = librosa.load(IDX[row["ytid"]], sr=SR, duration=10.0)

        if self.augment:
            if np.random.rand() < 0.5:
                y = y + np.random.randn(len(y)) * 0.005
            if np.random.rand() < 0.4:
                y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.9, 1.1))
            if np.random.rand() < 0.3:
                y = librosa.effects.pitch_shift(y, sr=SR, n_steps=np.random.randint(-2, 3))

        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        mel = np.pad(mel, ((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1] < TIME_FRAMES else mel[:, :TIME_FRAMES]
        return torch.tensor(mel[np.newaxis], dtype=torch.float32), torch.tensor(labels)


class CRNN(nn.Module):
    def __init__(self, n, sed=False):
        super().__init__()
        self.sed = sed
        self.cnn = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),  nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.lstm = nn.LSTM(32*16, 64, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(128, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

def mil_loss(fl, cl, crit): return crit(fl.max(1).values, cl)

_ds = AudioSetDS("audioset_v2_train.csv")
_x, _y = _ds[0]
print(f"Train: {len(_ds):,} clips | mel {tuple(_x.shape)} | labels {_y.numpy().astype(int)}")

Train: 9,922 clips | mel (1, 64, 128) | labels [0 0 0 0 0 0 1 0]


In [4]:
BATCH, LR, EPOCHS, SED_MODE = 32, 1e-3, 40, True
PATIENCE = 6

tr_loader = DataLoader(AudioSetDS("audioset_v2_train.csv", augment=True),
                       batch_size=BATCH, shuffle=True,  num_workers=8, pin_memory=True, persistent_workers=True)
va_loader = DataLoader(AudioSetDS("audioset_v2_val.csv",   augment=False),
                       batch_size=BATCH, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

dft = pd.read_csv("audioset_v2_train.csv")
pw  = [min((len(dft)-dft[c].sum())/max(dft[c].sum(),1), 50) for c in CLASSES]
pos_weight = torch.tensor(pw, dtype=torch.float32).to(DEVICE)

print("Batches/epoch:", len(tr_loader))
print("pos_weight:", {c: round(w,1) for c,w in zip(CLASSES, pw)})

Batches/epoch: 311
pos_weight: {'Scream': 9.9, 'Shout': 8.8, 'Crying': 8.8, 'Explosion': 8.7, 'Gunshot': 8.9, 'Glass': 14.4, 'Siren': 8.1, 'Alarm': 8.5}


In [5]:
model  = CRNN(8, sed=SED_MODE).to(DEVICE)
crit   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt    = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=3, factor=0.5)
scaler = torch.amp.GradScaler("cuda")

best_f1, pc, nb = 0.0, 0, len(tr_loader)
history = []

for ep in range(EPOCHS):
    model.train(); tl = 0
    for bi, (X, y) in enumerate(tr_loader):
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            out  = model(X)
            loss = mil_loss(out, y, crit) if SED_MODE else crit(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update()
        tl += loss.item()
        print(f"\rEp{ep+1:02d} | batch {bi+1:4d}/{nb} | loss {loss.item():.4f}", end="")
    tl /= nb

    model.eval(); P, L = [], []
    with torch.no_grad():
        for X, y in va_loader:
            with torch.amp.autocast("cuda"):
                lo = model(X.to(DEVICE))
                pr = torch.sigmoid(lo.max(1).values if SED_MODE else lo).float().cpu().numpy()
            P.append((pr > 0.5).astype(int)); L.append(y.numpy())
    P, L = np.vstack(P), np.vstack(L)
    mi = f1_score(L, P, average="micro", zero_division=0)
    ma = f1_score(L, P, average="macro", zero_division=0)
    history.append((ep+1, tl, mi, ma))
    print(f"\rEp{ep+1:02d} | loss {tl:.4f} | micro {mi:.4f} | macro {ma:.4f}                    ")
    sched.step(mi)

    if mi > best_f1:
        best_f1, pc = mi, 0
        torch.save(model.state_dict(), "best_audioset_sed_v2.pth")
        print(f"  --> saved (micro {best_f1:.4f})")
    else:
        pc += 1
        if pc >= PATIENCE:
            print(f"Early stop at ep{ep+1}"); break

print(f"\nDone. Best val micro-F1: {best_f1:.4f}   (v1 was 0.5631)")

Ep01 | batch    8/311 | loss 1.2100

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


Ep01 | batch   48/311 | loss 1.2107

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


Ep01 | loss 1.0140 | micro 0.4287 | macro 0.4320                    
  --> saved (micro 0.4287)


...

Ep13 | loss 0.7090 | micro 0.5382 | macro 0.5427                    
  --> saved (micro 0.5382)
Ep14 | batch  241/311 | loss 0.6224

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=512
  warnings.warn(


Ep14 | loss 0.6980 | micro 0.5336 | macro 0.5337                    
Ep15 | batch  261/311 | loss 0.5856

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=512
  warnings.warn(


Ep15 | loss 0.6999 | micro 0.5499 | macro 0.5471                    
  --> saved (micro 0.5499)
Ep16 | loss 0.6910 | micro 0.5507 | macro 0.5460                    
  --> saved (micro 0.5507)
Ep17 | loss 0.6783 | micro 0.5366 | macro 0.5395                    
Ep18 | loss 0.6669 | micro 0.5675 | macro 0.5614                    
  --> saved (micro 0.5675)
Ep19 | loss 0.6629 | micro 0.5420 | macro 0.5413                    
Ep20 | loss 0.6462 | micro 0.5523 | macro 0.5536                    
Ep21 | loss 0.6401 | micro 0.5545 | macro 0.5535                    
Ep22 | loss 0.6323 | micro 0.5546 | macro 0.5479                    
Ep23 | loss 0.6040 | micro 0.5617 | macro 0.5596                    
Ep24 | loss 0.5896 | micro 0.5589 | macro 0.5586                    
Early stop at ep24

Done. Best val micro-F1: 0.5675   (v1 was 0.5631)


In [6]:
model.load_state_dict(torch.load("best_audioset_sed_v2.pth"))
model.eval()

te_loader = DataLoader(AudioSetDS("audioset_v2_test.csv", augment=False),
                       batch_size=BATCH, shuffle=False, num_workers=8)

TP, TL = [], []
with torch.no_grad():
    for X, y in te_loader:
        with torch.amp.autocast("cuda"):
            lo = model(X.to(DEVICE))
            pr = torch.sigmoid(lo.max(1).values if SED_MODE else lo).float().cpu().numpy()
        TP.append(pr); TL.append(y.numpy())

test_probs  = np.vstack(TP).astype(np.float32)
test_labels = np.vstack(TL).astype(np.float32)
print("Test set:", test_probs.shape, "(eval_segments — never seen)\n")

# per-class threshold tuning
thr_grid = np.arange(0.10, 0.95, 0.01)
best_thr = {}
print(f"{'Class':12s} {'BestT':>6} {'F1@0.5':>8} {'F1tuned':>8} {'AP':>7}")
print("-"*48)
aps = []
for j, cls in enumerate(CLASSES):
    gt, sc = test_labels[:,j], test_probs[:,j]
    f1_50  = f1_score(gt, (sc>0.5).astype(int), zero_division=0)
    bf, bt = 0, 0.5
    for t in thr_grid:
        f1 = f1_score(gt, (sc>t).astype(int), zero_division=0)
        if f1 > bf: bf, bt = f1, t
    best_thr[cls] = round(float(bt),2)
    ap = average_precision_score(gt, sc); aps.append(ap)
    print(f"{cls:12s} {bt:6.2f} {f1_50:8.3f} {bf:8.3f} {ap:7.3f}")

tuned = np.zeros_like(test_probs)
for j,cls in enumerate(CLASSES):
    tuned[:,j] = (test_probs[:,j] > best_thr[cls]).astype(np.float32)

mi_flat  = f1_score(test_labels,(test_probs>0.5).astype(int),average="micro",zero_division=0)
mi_tuned = f1_score(test_labels,tuned,average="micro",zero_division=0)
ma_tuned = f1_score(test_labels,tuned,average="macro",zero_division=0)
mAP = np.mean(aps)

# bootstrap CI
boot=[]
for _ in range(5000):
    idx = np.random.choice(len(test_labels), len(test_labels), replace=True)
    boot.append(np.mean([average_precision_score(test_labels[idx,j], test_probs[idx,j])
                         for j in range(8) if test_labels[idx,j].sum()>0]))
lo, hi = np.percentile(boot,[2.5,97.5])

print(f"\n=== V2 TEST RESULTS (eval_segments) ===")
print(f"mAP            : {mAP:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"Micro-F1 flat  : {mi_flat:.4f}")
print(f"Micro-F1 tuned : {mi_tuned:.4f}")
print(f"Macro-F1 tuned : {ma_tuned:.4f}")
print(f"\nv1 was: mAP 0.549, micro 0.565, macro 0.575")

/tmp/ipykernel_1779001/426894072.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_audioset_sed_v2.pth"))
/user/HS400/as07181/minico

Test set: (1249, 8) (eval_segments — never seen)

Class         BestT   F1@0.5  F1tuned      AP
------------------------------------------------
Scream         0.70    0.293    0.407   0.284
Shout          0.45    0.547    0.554   0.566
Crying         0.71    0.615    0.646   0.698
Explosion      0.60    0.462    0.498   0.474
Gunshot        0.64    0.555    0.577   0.515
Glass          0.60    0.332    0.356   0.236
Siren          0.35    0.704    0.709   0.762
Alarm          0.60    0.658    0.676   0.743

=== V2 TEST RESULTS (eval_segments) ===
mAP            : 0.5348  95% CI [0.5100, 0.5703]
Micro-F1 flat  : 0.5427
Micro-F1 tuned : 0.5869
Macro-F1 tuned : 0.5531

v1 was: mAP 0.549, micro 0.565, macro 0.575
